In [ ]:
USE Northwinds2024Student;

### Exercise 1 Setup
Create a `dbo.Customers` staging table to work with

In [ ]:
DROP TABLE IF EXISTS dbo.Customers;

CREATE TABLE dbo.Customers
(
  CustomerId        INT           NOT NULL PRIMARY KEY,
  CustomerCompanyName NVARCHAR(40)  NOT NULL,
  CustomerCountry     NVARCHAR(15)  NOT NULL,
  CustomerRegion      NVARCHAR(15)  NULL,
  CustomerCity        NVARCHAR(15)  NOT NULL
);

### Exercise 1-1
Insert a single row into `dbo.Customers` with:
- CustomerId: 100
- CustomerCompanyName: Coho Winery
- CustomerCountry: USA
- CustomerRegion: WA
- CustomerCity: Redmond

In [ ]:
INSERT INTO dbo.Customers (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
VALUES (100, N'Coho Winery', N'USA', N'WA', N'Redmond');

SELECT * FROM dbo.Customers;

### Exercise 1-2
Insert into `dbo.Customers` all customers from `Sales.Customer` who have placed at least one order

In [ ]:
INSERT INTO dbo.Customers (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
  SELECT C.CustomerId, C.CustomerCompanyName, C.CustomerCountry, 
         C.CustomerRegion, C.CustomerCity
  FROM Sales.Customer AS C
  WHERE EXISTS
    (SELECT * FROM Sales.[Order] AS O
     WHERE O.CustomerId = C.CustomerId);

SELECT * FROM dbo.Customers;

### Exercise 1-3
Use `SELECT INTO` to create and populate `dbo.Orders` with orders placed in the years 2014 through 2016

In [ ]:
DROP TABLE IF EXISTS dbo.Orders;

SELECT *
INTO dbo.Orders
FROM Sales.[Order]
WHERE OrderDate >= '20140101'
  AND OrderDate < '20170101';

SELECT * FROM dbo.Orders;

### Exercise 2
Delete from `dbo.Orders` all orders placed before August 2015. Use `OUTPUT` to return the `OrderId` and `OrderDate` of deleted rows

In [ ]:
DELETE FROM dbo.Orders
  OUTPUT deleted.OrderId, deleted.OrderDate
WHERE OrderDate < '20150801';

### Exercise 3
Delete from `dbo.Orders` all orders placed by customers from Germany

In [ ]:
DELETE FROM dbo.Orders
WHERE EXISTS
  (SELECT *
   FROM dbo.Customers AS C
   WHERE dbo.Orders.CustomerId = C.CustomerId
     AND C.CustomerCountry = N'Germany');

SELECT * FROM dbo.Orders;

### Exercise 4
Update `dbo.Customers` and change all `NULL` `CustomerRegion` values to `<None>`. Use `OUTPUT` to show `CustomerId`, old, and new region

In [ ]:
UPDATE dbo.Customers
  SET CustomerRegion = '<None>'
OUTPUT
  deleted.CustomerId,
  deleted.CustomerRegion AS OldRegion,
  inserted.CustomerRegion AS NewRegion
WHERE CustomerRegion IS NULL;

### Exercise 5
Update `dbo.Orders` for all orders placed by customers from the UK. Set their shipping country, region, and city to match the corresponding customer's values from `dbo.Customers`

In [ ]:
UPDATE O
  SET O.ShipToAddress = C.CustomerCity
FROM dbo.Orders AS O
  INNER JOIN dbo.Customers AS C
    ON O.CustomerId = C.CustomerId
WHERE C.CustomerCountry = N'UK';

SELECT * FROM dbo.Orders;

### Exercise 6
Create `dbo.Orders` and `dbo.OrderDetails`, populate them, then truncate both tables handling the `FK` constraint

In [ ]:
DROP TABLE IF EXISTS dbo.OrderDetails, dbo.Orders;

CREATE TABLE dbo.Orders
(
  OrderId       INT           NOT NULL,
  CustomerId    INT           NULL,
  EmployeeId    INT           NOT NULL,
  OrderDate     DATE          NOT NULL,
  RequiredDate  DATE          NOT NULL,
  ShipToDate    DATE          NULL,
  ShipperId     INT           NOT NULL,
  Freight       MONEY         NOT NULL
    CONSTRAINT DFT_Orders_Freight DEFAULT(0),
  ShipToName    NVARCHAR(40)  NOT NULL,
  ShipToAddress NVARCHAR(60)  NOT NULL,
  CONSTRAINT PK_Orders PRIMARY KEY(OrderId)
);

CREATE TABLE dbo.OrderDetails
(
  OrderId             INT           NOT NULL,
  ProductId           INT           NOT NULL,
  UnitPrice           MONEY         NOT NULL
    CONSTRAINT DFT_OrderDetails_UnitPrice DEFAULT(0),
  Quantity            SMALLINT      NOT NULL
    CONSTRAINT DFT_OrderDetails_Qty DEFAULT(1),
  DiscountPercentage  NUMERIC(4,3)  NOT NULL
    CONSTRAINT DFT_OrderDetails_Discount DEFAULT(0),
  CONSTRAINT PK_OrderDetails PRIMARY KEY(OrderId, ProductId),
  CONSTRAINT FK_OrderDetails_Orders FOREIGN KEY(OrderId)
    REFERENCES dbo.Orders(OrderId),
  CONSTRAINT CHK_Discount   CHECK (DiscountPercentage BETWEEN 0 AND 1),
  CONSTRAINT CHK_Quantity   CHECK (Quantity > 0),
  CONSTRAINT CHK_UnitPrice  CHECK (UnitPrice >= 0)
);
GO

INSERT INTO dbo.Orders
  SELECT OrderId, CustomerId, EmployeeId, OrderDate, RequiredDate,
         ShipToDate, ShipperId, Freight, ShipToName, ShipToAddress
  FROM Sales.[Order];

INSERT INTO dbo.OrderDetails
  SELECT OrderId, ProductId, UnitPrice, Quantity, DiscountPercentage
  FROM Sales.OrderDetail;

-- Truncate both tables (must drop FK first, then re-add)
ALTER TABLE dbo.OrderDetails DROP CONSTRAINT FK_OrderDetails_Orders;

TRUNCATE TABLE dbo.OrderDetails;
TRUNCATE TABLE dbo.Orders;

ALTER TABLE dbo.OrderDetails ADD CONSTRAINT FK_OrderDetails_Orders
  FOREIGN KEY(OrderId) REFERENCES dbo.Orders(OrderId);

### Cleanup

In [ ]:
DROP TABLE IF EXISTS dbo.OrderDetails, dbo.Orders, dbo.Customers;

### P1: INSERT VALUES
A new customer needs to be added to the database. Insert a single customer row manually into `Sales.Customer`.

In [ ]:
INSERT INTO Sales.Customer (
    CustomerCompanyName, CustomerContactName, CustomerContactTitle,
    CustomerAddress, CustomerCity, CustomerCountry, CustomerPhoneNumber
)
VALUES (
    'Quick Stop Mart', 'Alex Johnson', 'Owner',
    '789 Broadway', 'Brooklyn', 'USA', '718-555-0234'
);

SELECT * FROM Sales.Customer WHERE CustomerCompanyName = 'Quick Stop Mart';

### P2: SELECT INTO
We need a backup of all orders placed in 2021. Use `SELECT INTO` to copy them into a new archive table.

In [ ]:
DROP TABLE IF EXISTS dbo.Orders2021Archive;

SELECT *
INTO dbo.Orders2021Archive
FROM Sales.[Order]
WHERE OrderDate >= '20210101'
  AND OrderDate < '20220101';

SELECT * FROM dbo.Orders2021Archive;

### P3: INSERT SELECT
We want a table of all products that cost more than $20. Use `INSERT SELECT` to pull them into a separate table.

In [ ]:
DROP TABLE IF EXISTS dbo.PremiumProducts;

SELECT ProductId, ProductName, UnitPrice, CategoryId, SupplierId
INTO dbo.PremiumProducts
FROM Production.Product
WHERE UnitPrice > 20;

SELECT * FROM dbo.PremiumProducts;

### P4: UPDATE basic
A shipping company changed their phone number. Update the phone number for Shipper ID 1 in `Sales.Shipper`.

In [ ]:
SELECT * FROM Sales.Shipper WHERE ShipperId = 1;

UPDATE Sales.Shipper
SET PhoneNumber = '503-555-1234'
WHERE ShipperId = 1;

SELECT * FROM Sales.Shipper WHERE ShipperId = 1;

### P5: UPDATE with JOIN
Update the freight cost on all orders for a specific customer by joining Orders to Customer to match on CustomerID.

In [ ]:
UPDATE O
SET O.Freight = O.Freight * 1.05
FROM Sales.[Order] O
JOIN Sales.Customer C ON O.CustomerId = C.CustomerId
WHERE C.CustomerCountry = 'USA';

SELECT O.OrderId, O.Freight, C.CustomerCompanyName
FROM Sales.[Order] O
JOIN Sales.Customer C ON O.CustomerId = C.CustomerId
WHERE C.CustomerCountry = 'USA';

### P6: UPDATE with OUTPUT
Raise all product prices in CategoryId = 2 by 8%. Use `OUTPUT` to show the old and new price for every product changed.

In [ ]:
UPDATE Production.Product
SET UnitPrice = UnitPrice * 1.08
OUTPUT
    inserted.ProductId,
    deleted.UnitPrice  AS OldPrice,
    inserted.UnitPrice AS NewPrice
WHERE CategoryId = 2;

### P7: DELETE with OUTPUT
Remove all orders placed before 2021 and use `OUTPUT` to log every deleted order before it's gone.

In [ ]:
DELETE FROM Sales.[Order]
OUTPUT
    deleted.OrderId,
    deleted.CustomerId,
    deleted.OrderDate
WHERE OrderDate < '20210101';

### P8: DELETE with subquery
Some order details reference products that no longer exist. Delete those orphaned order detail rows using a subquery.

In [ ]:
SELECT COUNT(*) AS OrphanedOrderDetails
FROM Sales.OrderDetail
WHERE ProductId NOT IN (SELECT ProductId FROM Production.Product);

DELETE FROM Sales.OrderDetail
WHERE ProductId NOT IN (
    SELECT ProductId FROM Production.Product
);

### P9: MERGE
We received updated product prices and some new products. `MERGE` updates existing ones and inserts new ones without duplicates.

In [ ]:
DROP TABLE IF EXISTS dbo.ProductStage;

CREATE TABLE dbo.ProductStage (
    ProductId    INT,
    ProductName  NVARCHAR(40),
    UnitPrice    MONEY,
    SupplierId   INT,
    CategoryId   INT,
    Discontinued BIT
);

INSERT INTO dbo.ProductStage VALUES
    (1,   'Product HHYDP Updated', 18.00, 1, 1, 0),
    (999, 'Brand New Product',     35.00, 1, 2, 0);

MERGE INTO Production.Product AS TGT
USING dbo.ProductStage AS SRC
    ON TGT.ProductId = SRC.ProductId
WHEN MATCHED THEN
    UPDATE SET TGT.UnitPrice = SRC.UnitPrice
WHEN NOT MATCHED THEN
    INSERT (ProductName, UnitPrice, SupplierId, CategoryId, Discontinued)
    VALUES (SRC.ProductName, SRC.UnitPrice, SRC.SupplierId, SRC.CategoryId, SRC.Discontinued);

SELECT * FROM Production.Product WHERE ProductId IN (1, 999);

### P10: Nested DML
Raise prices for Category 3 products by 20% and log only the ones that crossed the $50 mark into an audit table.

In [ ]:
DROP TABLE IF EXISTS dbo.PriceAudit;

CREATE TABLE dbo.PriceAudit (
    LSN       INT IDENTITY PRIMARY KEY,
    TS        DATETIME2 DEFAULT SYSDATETIME(),
    ProductId INT,
    OldPrice  MONEY,
    NewPrice  MONEY
);

INSERT INTO dbo.PriceAudit (ProductId, OldPrice, NewPrice)
    SELECT ProductId, OldPrice, NewPrice
    FROM (
        UPDATE Production.Product
            SET UnitPrice = UnitPrice * 1.20
        OUTPUT
            inserted.ProductId,
            deleted.UnitPrice  AS OldPrice,
            inserted.UnitPrice AS NewPrice
        WHERE CategoryId = 3
    ) AS D
    WHERE OldPrice < 50.0 AND NewPrice >= 50.0;

SELECT * FROM dbo.PriceAudit;